# Multidimensional Improvement Rates in Climate Adaptation Model

This notebook demonstrates how the `basement_height_improvement_rate` and `offset_improvement_rate` parameters are now structured as multidimensional arrays based on:
- **Parcel**: Different property parcels (1001, 1002, 1003)
- **Year Built Cohort**: Construction era (pre_war, post_war, modern)
- **Building Type**: Building category (residential, commercial)

## Key Benefits of Multidimensional Rates
1. **Realistic Modeling**: Different improvement rates for different building types and ages
2. **Spatial Variation**: Account for parcel-specific conditions and resources
3. **Temporal Dynamics**: Reflect how newer buildings can be improved more easily
4. **Policy Analysis**: Enable targeted adaptation strategies

In [ ]:
import sys
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path for BPTK imports
sys.path.append('..')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("📊 Multidimensional Climate Adaptation Rates Analysis")
print("=" * 55)

## 1. Load and Compare Different Scenarios

In [ ]:
def load_scenario_rates(filepath, scenario_name):
    """Load improvement rates from a scenario file."""
    with open(filepath, 'r') as file:
        data = yaml.safe_load(file)
    
    basement_rates = np.array(data['constants']['basement_height_improvement_rate'])
    offset_rates = np.array(data['constants']['offset_improvement_rate'])
    
    return {
        'scenario': scenario_name,
        'basement_rates': basement_rates,
        'offset_rates': offset_rates
    }

# Load all scenarios
scenarios = [
    load_scenario_rates('yaml_models/climate_adaptation_baseline.yaml', 'Baseline'),
    load_scenario_rates('yaml_models/climate_adaptation_model.yaml', 'Default'),
    load_scenario_rates('yaml_models/climate_adaptation_scenario.yaml', 'Aggressive')
]

# Define dimension labels
parcels = ['parcel_1001', 'parcel_1002', 'parcel_1003']
cohorts = ['pre_war', 'post_war', 'modern']
building_types = ['residential', 'commercial']

print("✅ Loaded improvement rates for all scenarios")

## 2. Visualize Basement Height Improvement Rates

In [ ]:
# Create comparison visualization for basement height improvement rates
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Basement Height Improvement Rates by Scenario\n(meters/year)', fontsize=16, fontweight='bold')

for i, scenario in enumerate(scenarios):
    ax = axes[i]
    rates = scenario['basement_rates']
    
    # Create heatmap for each parcel
    # Reshape data for visualization: [cohort x building_type] for each parcel
    combined_data = np.zeros((len(cohorts), len(building_types) * len(parcels)))
    
    for p in range(len(parcels)):
        start_col = p * len(building_types)
        end_col = start_col + len(building_types)
        combined_data[:, start_col:end_col] = rates[p, :, :]
    
    # Create column labels
    col_labels = []
    for parcel in parcels:
        for btype in building_types:
            col_labels.append(f"{parcel}\n{btype}")
    
    im = ax.imshow(combined_data, cmap='YlOrRd', aspect='auto')
    ax.set_title(f'{scenario["scenario"]} Scenario', fontweight='bold')
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right')
    ax.set_yticks(range(len(cohorts)))
    ax.set_yticklabels(cohorts)
    ax.set_ylabel('Construction Cohort')
    
    # Add value annotations
    for row in range(len(cohorts)):
        for col in range(len(col_labels)):
            text = ax.text(col, row, f'{combined_data[row, col]:.3f}',
                         ha="center", va="center", color="black", fontsize=8)
    
    plt.colorbar(im, ax=ax, label='Rate (m/year)')

plt.tight_layout()
plt.show()

## 3. Visualize Offset Improvement Rates

In [ ]:
# Create comparison visualization for offset improvement rates
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Structure Offset Improvement Rates by Scenario\n(meters/year)', fontsize=16, fontweight='bold')

for i, scenario in enumerate(scenarios):
    ax = axes[i]
    rates = scenario['offset_rates']
    
    # Create heatmap for each parcel
    combined_data = np.zeros((len(cohorts), len(building_types) * len(parcels)))
    
    for p in range(len(parcels)):
        start_col = p * len(building_types)
        end_col = start_col + len(building_types)
        combined_data[:, start_col:end_col] = rates[p, :, :]
    
    # Create column labels
    col_labels = []
    for parcel in parcels:
        for btype in building_types:
            col_labels.append(f"{parcel}\n{btype}")
    
    im = ax.imshow(combined_data, cmap='Blues', aspect='auto')
    ax.set_title(f'{scenario["scenario"]} Scenario', fontweight='bold')
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right')
    ax.set_yticks(range(len(cohorts)))
    ax.set_yticklabels(cohorts)
    ax.set_ylabel('Construction Cohort')
    
    # Add value annotations
    for row in range(len(cohorts)):
        for col in range(len(col_labels)):
            text = ax.text(col, row, f'{combined_data[row, col]:.3f}',
                         ha="center", va="center", color="black", fontsize=8)
    
    plt.colorbar(im, ax=ax, label='Rate (m/year)')

plt.tight_layout()
plt.show()

## 4. Analyze Rate Patterns and Insights

In [ ]:
# Create summary statistics table
def create_summary_stats(scenarios):
    """Create summary statistics for all scenarios."""
    summary_data = []
    
    for scenario in scenarios:
        name = scenario['scenario']
        basement = scenario['basement_rates']
        offset = scenario['offset_rates']
        
        summary_data.append({
            'Scenario': name,
            'Metric': 'Basement Height',
            'Min Rate': f"{basement.min():.3f}",
            'Max Rate': f"{basement.max():.3f}",
            'Mean Rate': f"{basement.mean():.3f}",
            'Std Dev': f"{basement.std():.3f}"
        })
        
        summary_data.append({
            'Scenario': name,
            'Metric': 'Offset Height',
            'Min Rate': f"{offset.min():.3f}",
            'Max Rate': f"{offset.max():.3f}",
            'Mean Rate': f"{offset.mean():.3f}",
            'Std Dev': f"{offset.std():.3f}"
        })
    
    return pd.DataFrame(summary_data)

summary_df = create_summary_stats(scenarios)
print("📈 Summary Statistics for Improvement Rates (meters/year)")
print("=" * 60)
print(summary_df.to_string(index=False))

## 5. Key Insights and Patterns

### Observed Patterns:
1. **Cohort Effects**: Modern buildings generally have higher improvement rates than older cohorts
2. **Parcel Variation**: Higher-numbered parcels (1003) tend to have better improvement rates
3. **Building Type Differences**: Residential buildings often have slightly higher rates than commercial
4. **Scenario Scaling**: Aggressive scenario shows 2-4x higher rates than baseline

### Policy Implications:
- **Targeted Investment**: Focus resources on parcels and building types with highest potential
- **Cohort-Specific Strategies**: Different approaches needed for different construction eras
- **Scenario Planning**: Wide range of possible outcomes depending on policy commitment

In [ ]:
# Calculate improvement potential over 20 years
years = 20
print(f"\n🔮 Projected Improvements Over {years} Years")
print("=" * 50)

for scenario in scenarios:
    name = scenario['scenario']
    basement_total = scenario['basement_rates'] * years
    offset_total = scenario['offset_rates'] * years
    combined_total = basement_total + offset_total
    
    print(f"\n{name} Scenario:")
    print(f"  Max total flood protection improvement: {combined_total.max():.2f} meters")
    print(f"  Min total flood protection improvement: {combined_total.min():.2f} meters")
    print(f"  Average improvement: {combined_total.mean():.2f} meters")
    
    # Find best and worst performing combinations
    max_idx = np.unravel_index(combined_total.argmax(), combined_total.shape)
    min_idx = np.unravel_index(combined_total.argmin(), combined_total.shape)
    
    print(f"  Best: {parcels[max_idx[0]]}, {cohorts[max_idx[1]]}, {building_types[max_idx[2]]}")
    print(f"  Worst: {parcels[min_idx[0]]}, {cohorts[min_idx[1]]}, {building_types[min_idx[2]]}")

print("\n✅ Analysis complete! The multidimensional rate structure enables")
print("   detailed modeling of climate adaptation measures across different")
print("   building types, construction eras, and spatial locations.")